# Synthetic versus direct: how much volatility does a constructed cross add?

A cap built on the direct[1h] method needs a price series for the traded pair. Most pairs CoW sees have no direct Binance book, so the series has to be synthetic: divide the two USDT legs and treat the ratio as the cross. Earlier work found this safe for genuine crosses but wrong for correlated pairs, where the real WBETH/ETH cap came out at 0.015 bps against 6.16 bps for its synthetic. That rested on a single pair measured one way. This notebook re-tests it and adds two more pairs, with a confidence interval on each gap.

SOL/ETH is a genuine cross that Binance lists directly, so both series come from one venue at one sampling rate and any difference is due to the construction alone. WBETH/ETH is the correlated pair the original finding came from, and it is the only one here with a direct listing, a synthetic and an on chain price all at once. COW/ETH is the case that matters most for CoW and is the hardest, because Binance has no COW/ETH book at all, so its direct series has to come from a Uniswap V3 pool and the venue difference cannot be separated from the construction.

The synthetic log return is the difference of the two leg returns, so its variance is the sum of the leg variances minus twice their covariance. Independent noise in the legs does not cancel in that difference, it adds. A synthetic therefore overstates volatility when the legs' noise is uncorrelated, and overstates most when the true cross is quiet. Each section reports the leg correlation the synthetic would need to match the direct series against the correlation the legs actually have.

In [ ]:
import os, io, json, time, zipfile, hashlib, warnings, urllib.request
import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
warnings.filterwarnings("ignore")

DATE_START, DATE_END = "2026-05-22", "2026-07-21"     # matches the clustering notebooks
T_EXCL   = 26                                          # exclusivity window (s), for the cap view
N_BOOT   = 2000                                        # stationary-bootstrap replicates
MEAN_BLOCK = 60                                        # expected block length (observations)
POOL_COW = "0xfbb81382cce6b9ce58f8645353f70d9db1bb69af"   # deepest Uniswap V3 COW/WETH pool (ethereum)
POOL_COW_FEE_BPS = 30                                      # 0.30% tier (PoolCreated fee=3000)
POOL_WBETH = "0x379044e32f5a162233e82de19da852255d0951b8" # deepest wBETH/ETH pool: PancakeSwap V3 on BNB
POOL_WBETH_FEE_BPS = 5                                     # 0.05% tier (PoolCreated fee=500)
SEED = 7

DAYS = [d.strftime("%Y-%m-%d") for d in pd.date_range(DATE_START, DATE_END)]
# cache root: the repo's data/ when run from notebooks/, otherwise a local clone
_cwd = os.path.abspath(os.getcwd())
ROOT = (os.path.abspath(os.path.join(_cwd, os.pardir, "data")) if os.path.basename(_cwd) == "notebooks"
        else os.path.expanduser("~/dev/penalty-research/data"))
CACHE_1M   = os.path.join(ROOT, "binance_klines/1m")
CACHE_DUNE = os.path.join(ROOT, "dune")
for p in (CACHE_1M, CACHE_DUNE): os.makedirs(p, exist_ok=True)

GRAY, INK, GRID, SURF, AXIS = "#898781", "#0b0b0b", "#e1e0d9", "#fcfcfb", "#c3c2b7"
C_DIR, C_SYN = "#3d7a78", "#e0a63a"      # direct = teal, synthetic = amber (as in the cap notebooks)
C_LEG_A, C_LEG_B = "#2a78d6", "#0b0b0b"

def style(fig, w=1050, h=430, title=None):
    fig.update_layout(width=w, height=h, template="plotly_white", title=title,
                      font=dict(family="Inter, Helvetica, Arial", size=12, color=INK),
                      paper_bgcolor=SURF, plot_bgcolor=SURF, margin=dict(l=70, r=30, t=80, b=55))
    fig.update_xaxes(showgrid=True, gridcolor=GRID, linecolor=AXIS, zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor=GRID, linecolor=AXIS, zeroline=False)
    return fig

def dune_key():
    """DUNE_API_KEY if set, else the key file the fetch scripts already use."""
    k = os.environ.get("DUNE_API_KEY")
    if k: return k.strip()
    path = os.path.expanduser("~/.config/cow_dune/api_key")
    if os.path.exists(path): return open(path).read().strip()
    raise RuntimeError("no Dune credential: set DUNE_API_KEY or write the key to "
                       "~/.config/cow_dune/api_key")

RENDER = "plotly_mimetype+notebook"
rng = np.random.default_rng(SEED)
print(f"window {DATE_START} .. {DATE_END} ({len(DAYS)} days) | exclusivity {T_EXCL}s")

## 1. Binance 1 minute data

Six symbols: the direct listings SOLETH and WBETHETH, the numerators COWUSDT and WBETHUSDT, and the shared legs SOLUSDT and ETHUSDT. Monthly zips for complete months, daily zips for the partial month, cached with atomic writes in the directory the cap notebooks already use. Each file is reduced to a 1 minute close series on load.

In [ ]:
def files_1m(sym):
    out = []
    for mo in ("2026-05", "2026-06"):
        out.append((f"https://data.binance.vision/data/spot/monthly/klines/{sym}/1m/{sym}-1m-{mo}.zip",
                    os.path.join(CACHE_1M, f"{sym}-1m-{mo}.zip")))
    for d in [x for x in DAYS if x >= "2026-07-01"]:
        out.append((f"https://data.binance.vision/data/spot/daily/klines/{sym}/1m/{sym}-1m-{d}.zip",
                    os.path.join(CACHE_1M, f"{sym}-1m-{d}.zip")))
    return out

def fetch(url, path):
    if os.path.exists(path) or os.path.exists(path + ".404"): return
    try:
        with urllib.request.urlopen(url, timeout=120) as r: blob = r.read()
        tmp = path + ".part"
        with open(tmp, "wb") as fh: fh.write(blob)
        os.replace(tmp, path)                      # atomic: never a half file for a parallel reader
    except Exception:
        open(path + ".404", "w").close()

def closes_1m(sym):
    frames = []
    for url, path in files_1m(sym):
        fetch(url, path)
        if not os.path.exists(path): continue
        try:
            with zipfile.ZipFile(path) as z:
                name = z.namelist()[0]
                hh = not z.open(name).read(16)[:1].isdigit()          # some files carry a header row
                frames.append(pd.read_csv(z.open(name), header=0 if hh else None,
                                          usecols=[0, 4], names=["ts", "close"]))
        except Exception:
            continue
    df = pd.concat(frames).dropna()
    unit = "us" if df["ts"].iloc[0] > 10**14 else "ms"                # Binance switched to microseconds in 2025
    s = pd.Series(df["close"].to_numpy(float),
                  index=pd.to_datetime(df["ts"], unit=unit)).sort_index()
    s = s[~s.index.duplicated(keep="last")]
    return s[(s.index >= DATE_START) & (s.index < pd.Timestamp(DATE_END) + pd.Timedelta(days=1))]

t0 = time.time()
PX = {}
for sym in ("SOLETH", "SOLUSDT", "ETHUSDT", "COWUSDT", "WBETHETH", "WBETHUSDT"):
    PX[sym] = closes_1m(sym)
    print(f"  {sym:9s} {len(PX[sym]):>7,} minutes  {PX[sym].index[0]} .. {PX[sym].index[-1]}  ({time.time()-t0:.0f}s)")

# synthetics: the cross implied by dividing two USDT legs
PX["SOL/ETH syn"]   = (PX["SOLUSDT"]/PX["ETHUSDT"]).dropna()
PX["COW/ETH syn"]   = (PX["COWUSDT"]/PX["ETHUSDT"]).dropna()
PX["WBETH/ETH syn"] = (PX["WBETHUSDT"]/PX["ETHUSDT"]).dropna()
print(f"\nSOL/ETH   direct {PX['SOLETH'].iloc[-1]:.6f} vs synthetic {PX['SOL/ETH syn'].iloc[-1]:.6f}")
print(f"WBETH/ETH direct {PX['WBETHETH'].iloc[-1]:.6f} vs synthetic {PX['WBETH/ETH syn'].iloc[-1]:.6f}")
print(f"COW/ETH   synthetic {PX['COW/ETH syn'].iloc[-1]:.3e} (no direct Binance listing)")

## 2. On chain prices

Two pools are needed. COW/ETH comes from the deepest Uniswap V3 COW/WETH pool on Ethereum, which carries 8,354 swaps and 3.1 million dollars over the window, more than four times the next pool. WBETH/ETH comes from PancakeSwap V3 on BNB, with 7,432 swaps, because no Uniswap wBETH/ETH pool has usable volume on any chain.

Price is taken from sqrtPriceX96, the pool's marginal price after each swap, rather than from executed amounts, which embed the fee and each trade's size dependent slippage. Both pools hold two 18 decimal tokens, so the square of sqrtPriceX96 divided by 2^96 is token1 per token0, and its reciprocal gives the quote convention Binance uses.

Two limits carry through the analysis. The marginal price after a swap moves in the direction of that swap, which gives each on chain series its own noise. More importantly, a pool is only corrected toward the wider market once the mispricing exceeds its fee plus gas, so its fee sets a floor on the movement it can resolve. That floor is 30 bps for the COW pool and 5 bps for the wBETH pool, against caps of single digit bps over 26 seconds. Both pools are therefore coarse relative to the quantity being measured, the COW one severely so.

In [ ]:
def pool_prices(table, pool, label, invert=True):
    """Marginal pool price after each swap, from sqrtPriceX96. Both tokens carry 18 decimals in
    both pools used here, so no decimal adjustment is needed. invert flips token1-per-token0
    into the quote convention Binance uses for the same pair."""
    sql = f"""
    select evt_block_time, evt_index, sqrtPriceX96
    from {table}
    where contract_address = {pool}
      and evt_block_time >= timestamp '{DATE_START}'
      and evt_block_time <  timestamp '{DATE_END}' + interval '1' day
    order by evt_block_time, evt_index
    """
    slug = hashlib.md5(sql.encode()).hexdigest()[:12]
    path = os.path.join(CACHE_DUNE, f"{label}_{slug}.pkl")
    if os.path.exists(path):
        raw, src = pd.read_pickle(path), "cache"
    else:
        from dune_client.client import DuneClient
        dc = DuneClient(dune_key(), request_timeout=900)
        raw = pd.DataFrame(dc.run_sql(query_sql=sql, performance="medium").get_rows())
        raw.to_pickle(path)
        src = "Dune"
    raw = raw.copy()
    raw["evt_block_time"] = pd.to_datetime(raw["evt_block_time"]).dt.tz_localize(None)
    p = (raw.sqrtPriceX96.astype(float)/2**96)**2
    raw["px"] = 1.0/p if invert else p
    raw = raw.sort_values(["evt_block_time", "evt_index"])
    # one price per timestamp: the last swap in a block is that block's closing pool price
    s = raw.groupby("evt_block_time").px.last()
    gap = np.diff(s.index.values).astype("timedelta64[s]").astype(float)/60
    print(f"{label}: {len(s):,} priced blocks from {src} | {len(s)/len(DAYS):.0f} per day | "
          f"gap median {np.median(gap):.1f} min, mean {gap.mean():.1f}, p90 {np.percentile(gap, 90):.1f}")
    return s

PX["COW/ETH onchain"]   = pool_prices("uniswap_v3_ethereum.Pair_evt_Swap",
                                      POOL_COW, "cow_weth_univ3_eth")
PX["WBETH/ETH onchain"] = pool_prices("pancakeswap_v3_bnb.PancakeV3Pool_evt_Swap",
                                      POOL_WBETH, "wbeth_eth_pcsv3_bnb")
print(f"\n  COW/ETH   range {PX['COW/ETH onchain'].min():.3e} .. {PX['COW/ETH onchain'].max():.3e} ETH per COW"
      f"  (pool fee {POOL_COW_FEE_BPS} bps)")
print(f"  WBETH/ETH range {PX['WBETH/ETH onchain'].min():.4f} .. {PX['WBETH/ETH onchain'].max():.4f} ETH per wBETH"
      f"  (pool fee {POOL_WBETH_FEE_BPS} bps)")

## 3. SOL/ETH, the controlled case

Both series come from Binance at 1 minute sampling, so venue, clock and microstructure are held fixed and only the construction differs. The top panel overlays the two series, the bottom shows the legs rebased to 100. The levels should track closely, since arbitrage keeps the triangle closed. What the level chart cannot show is the difference in short horizon movement, which is what a cap is calibrated on.

In [ ]:
def save_html(fig, name):
    """Standalone interactive file: plotly.js is embedded so it opens offline."""
    path = os.path.join(os.getcwd(), name)
    fig.write_html(path, include_plotlyjs=True, full_html=True)
    print(f"interactive HTML: {path} ({os.path.getsize(path)/1e6:.1f} MB)")

def plot_pair(direct_key, syn_key, leg_a, leg_b, rate_label, title, days=30, step=3, html=None):
    last = pd.Timestamp(DAYS[-days])
    keys = [k for k in (direct_key, syn_key, leg_a, leg_b) if k is not None]
    CH = {k: PX[k][PX[k].index >= last].iloc[::step] for k in keys}
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        subplot_titles=[f"{rate_label}: direct vs synthetic ratio",
                                        f"the legs: {leg_a} and {leg_b}"])
    fig.add_trace(go.Scatter(x=CH[direct_key].index, y=CH[direct_key].to_numpy(),
                             name=f"{direct_key} (direct)",
                             line=dict(color=C_DIR, width=1.4)), row=1, col=1)
    fig.add_trace(go.Scatter(x=CH[syn_key].index, y=CH[syn_key].to_numpy(),
                             name=f"synthetic ({leg_a} / {leg_b})",
                             line=dict(color=C_SYN, width=1)), row=1, col=1)
    # legs are rebased to 100 rather than given a second axis: their levels differ by orders of
    # magnitude, and what matters here is that they move together, which is what the cross cancels
    for leg, col in ((leg_a, C_LEG_A), (leg_b, C_LEG_B)):
        fig.add_trace(go.Scatter(x=CH[leg].index, y=CH[leg].to_numpy()/CH[leg].iloc[0]*100,
                                 name=f"{leg} (rebased)", line=dict(color=col, width=1)),
                      row=2, col=1)
    fig.update_yaxes(title_text=rate_label, row=1, col=1)
    fig.update_yaxes(title_text="legs rebased to 100", row=2, col=1)
    style(fig, w=1150, h=650, title=title)
    fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
    fig.show(renderer=RENDER)
    if html: save_html(fig, html)
    print(f"points per trace: {len(CH[direct_key])} direct / {len(CH[syn_key])} synthetic "
          f"({step} minute sampling, last {days} days)")

plot_pair("SOLETH", "SOL/ETH syn", "SOLUSDT", "ETHUSDT", "SOL/ETH rate",
          "Real vs synthetic SOL/ETH, last 30 days (3 minute sampling)",
          html="chart_soleth_direct_vs_synthetic.html")

## 4. Measuring the gap

Three estimators are used.

The volatility ratio is the headline: the standard deviation of synthetic log returns divided by that of direct log returns on a common grid. A ratio of 2 means a cap calibrated on the synthetic is roughly twice as loose as the pair warrants.

The confidence interval comes from a stationary bootstrap, which resamples blocks of random geometric length and so preserves the serial dependence in returns. Both series are resampled with the same indices, since they are two views of one asset rather than two independent samples. A classical test for equal variances assumes independent normal draws, so the Brown Forsythe statistic is reported only as a cross check.

The covariance decomposition explains the result. Given the observed direct variance, the variance identity can be inverted for the leg correlation the synthetic would need in order to match. Comparing that with the correlation the legs actually show says whether the excess is arithmetic or something else.

In [ ]:
def sb_indices(n, mean_block, size, rng):
    """Stationary bootstrap (Politis-Romano 1994), vectorised: geometric blocks, wrapped."""
    p = 1.0/mean_block
    new = rng.random(size) < p
    new[0] = True
    starts = rng.integers(0, n, size=size)
    anchor = np.maximum.accumulate(np.where(new, np.arange(size), 0))
    return (starts[anchor] + (np.arange(size) - anchor)) % n

def vol_ratio(r_syn, r_dir):
    return float(np.sqrt(np.sum(r_syn**2)/np.sum(r_dir**2)))

def compare(r_syn, r_dir, label, mean_block=MEAN_BLOCK, n_boot=N_BOOT, seed=SEED):
    """Paired comparison of two aligned return series. Returns a one-row summary."""
    r_syn, r_dir = np.asarray(r_syn, float), np.asarray(r_dir, float)
    ok = np.isfinite(r_syn) & np.isfinite(r_dir)
    r_syn, r_dir = r_syn[ok], r_dir[ok]
    n = len(r_syn)
    point = vol_ratio(r_syn, r_dir)
    g = np.random.default_rng(seed)
    boot = np.empty(n_boot)
    for b in range(n_boot):
        idx = sb_indices(n, mean_block, n, g)          # SAME idx for both: paired resample
        boot[b] = vol_ratio(r_syn[idx], r_dir[idx])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    p_boot = 2*min((boot <= 1).mean(), (boot >= 1).mean())   # two-sided, H0: ratio = 1
    bf = stats.levene(r_syn, r_dir, center="median")          # Brown-Forsythe
    return dict(case=label, n=n,
                vol_syn_bps=float(np.std(r_syn)*1e4), vol_dir_bps=float(np.std(r_dir)*1e4),
                ratio=point, ci_lo=float(lo), ci_hi=float(hi),
                p_bootstrap=float(max(p_boot, 1/n_boot)), p_brown_forsythe=float(bf.pvalue))

def aligned_returns(direct_key, syn_key, minutes):
    """Log returns of both series on a shared regular grid."""
    a = PX[direct_key].resample(f"{minutes}min").last()
    b = PX[syn_key].resample(f"{minutes}min").last()
    df = pd.concat([a.rename("d"), b.rename("s")], axis=1).dropna()
    df = df[(df.d > 0) & (df.s > 0)]
    return np.diff(np.log(df.s.to_numpy())), np.diff(np.log(df.d.to_numpy()))

rs, rd = aligned_returns("SOLETH", "SOL/ETH syn", 1)
res_sol = compare(rs, rd, "SOL/ETH @ 1 minute")
print(pd.DataFrame([res_sol]).round(4).to_string(index=False))
print(f"\nsynthetic is {res_sol['ratio']:.2f}x the direct volatility "
      f"(95% CI {res_sol['ci_lo']:.2f} to {res_sol['ci_hi']:.2f})")

## 5. The horizon sweep

A single sampling frequency cannot separate two explanations for an elevated ratio. Independent leg noise is a fixed variance per observation, so it dominates at fast sampling and washes out as the interval lengthens. Real movement that the direct book misses would persist at every horizon.

The left panel plots volatility rescaled to a per minute basis, where a series free of microstructure noise traces a flat line. The right panel plots the ratio itself. Decay toward 1 indicates noise.

In [ ]:
HORIZONS = [1, 2, 5, 10, 15, 30, 60, 120, 240]

def sweep(direct_key, syn_key, label):
    rows = []
    for m in HORIZONS:
        rs, rd = aligned_returns(direct_key, syn_key, m)
        if len(rs) < 50: continue
        rows.append(dict(minutes=m, n=len(rs),
                         syn_bps_per_sqrt_min=np.std(rs)*1e4/np.sqrt(m),
                         dir_bps_per_sqrt_min=np.std(rd)*1e4/np.sqrt(m),
                         ratio=vol_ratio(rs, rd)))
    return pd.DataFrame(rows).set_index("minutes")

sw_sol = sweep("SOLETH", "SOL/ETH syn", "SOL/ETH")
display(sw_sol.round(3))

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "volatility rescaled per minute (flat = no microstructure noise)",
    "synthetic / direct volatility ratio"])
fig.add_trace(go.Scatter(x=sw_sol.index, y=sw_sol.dir_bps_per_sqrt_min, name="SOLETH direct",
                         mode="lines+markers", line=dict(color=C_DIR, width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=sw_sol.index, y=sw_sol.syn_bps_per_sqrt_min, name="synthetic",
                         mode="lines+markers", line=dict(color=C_SYN, width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=sw_sol.index, y=sw_sol.ratio, name="ratio", showlegend=False,
                         mode="lines+markers", line=dict(color=INK, width=2)), row=1, col=2)
fig.add_hline(y=1.0, line=dict(color=GRAY, width=1, dash="dot"), row=1, col=2)
fig.update_xaxes(type="log", title_text="sampling interval (minutes)")
fig.update_yaxes(title_text="bps per sqrt(minute)", row=1, col=1)
fig.update_yaxes(title_text="ratio", row=1, col=2)
style(fig, w=1200, h=420, title="SOL/ETH: does the synthetic excess survive coarser sampling?")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
fig.show(renderer=RENDER)
print(f"ratio at 1 minute {sw_sol.ratio.iloc[0]:.2f} -> at {sw_sol.index[-1]} minutes "
      f"{sw_sol.ratio.iloc[-1]:.2f}")

## 6. COW/ETH, on chain against synthetic

The same view with the direct series taken from the Uniswap pool. The on chain line is stepped because the pool reprices only when someone trades it, roughly every ten minutes, while the synthetic moves every minute.

Comparing the two on a shared 1 minute grid would understate on chain volatility, since most 1 minute on chain returns are mechanically zero. The next cell samples the synthetic at the on chain timestamps instead, so both series are observed at the same instants, and normalises by the square root of elapsed time to handle the irregular spacing.

In [ ]:
last = pd.Timestamp(DAYS[-30])
oc = PX["COW/ETH onchain"][PX["COW/ETH onchain"].index >= last]
sy = PX["COW/ETH syn"][PX["COW/ETH syn"].index >= last].iloc[::3]
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=["COW/ETH: Uniswap V3 pool price vs Binance synthetic ratio",
                                    "the legs: COWUSDT and ETHUSDT"])
fig.add_trace(go.Scatter(x=oc.index, y=oc.to_numpy(), name="Uniswap V3 COW/WETH (direct)",
                         line=dict(color=C_DIR, width=1.4)), row=1, col=1)
fig.add_trace(go.Scatter(x=sy.index, y=sy.to_numpy(), name="synthetic (COWUSDT / ETHUSDT)",
                         line=dict(color=C_SYN, width=1)), row=1, col=1)
cu = PX["COWUSDT"][PX["COWUSDT"].index >= last].iloc[::3]
eu = PX["ETHUSDT"][PX["ETHUSDT"].index >= last].iloc[::3]
for leg, ser, col in (("COWUSDT", cu, C_LEG_A), ("ETHUSDT", eu, C_LEG_B)):
    fig.add_trace(go.Scatter(x=ser.index, y=ser.to_numpy()/ser.iloc[0]*100,
                             name=f"{leg} (rebased)", line=dict(color=col, width=1)), row=2, col=1)
fig.update_yaxes(title_text="ETH per COW", row=1, col=1)
fig.update_yaxes(title_text="legs rebased to 100", row=2, col=1)
style(fig, w=1150, h=650, title="On-chain vs synthetic COW/ETH, last 30 days")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
fig.show(renderer=RENDER)
save_html(fig, "chart_coweth_onchain_vs_synthetic.html")
print(f"points: {len(oc)} on-chain (irregular) / {len(sy)} synthetic (3 minute grid)")

### Matched timestamp comparison

For each pair of consecutive on chain prices the synthetic is read at the same two instants, using the last completed minute at or before each, and both returns are divided by the square root of the elapsed minutes. Intervals shorter than thirty seconds or longer than two hours are dropped. The former are near same block repricings that carry no information, the latter are gaps over which the two venues drift apart for unrelated reasons.

In [ ]:
MIN_GAP, MAX_GAP = 0.5, 120.0          # minutes

def matched(onchain_key, other_key, label, verbose=True):
    """Compare an irregular on-chain series against a 1 minute series sampled at the same
    instants. Returns the per-sqrt-minute returns, the test result and the gap sweep."""
    oc = PX[onchain_key]
    at = PX[other_key].reindex(oc.index, method="ffill")          # last completed minute
    df = pd.concat([oc.rename("d"), at.rename("s")], axis=1).dropna()
    df = df[(df.d > 0) & (df.s > 0)]
    dt  = np.diff(df.index.values).astype("timedelta64[s]").astype(float)/60.0
    r_d = np.diff(np.log(df.d.to_numpy()))
    r_s = np.diff(np.log(df.s.to_numpy()))
    keep = (dt >= MIN_GAP) & (dt <= MAX_GAP)
    r_d, r_s, dt = r_d[keep], r_s[keep], dt[keep]
    # normalise to a per-sqrt-minute basis so irregular spacing does not drive the comparison
    r_d_n, r_s_n = r_d/np.sqrt(dt), r_s/np.sqrt(dt)
    res = compare(r_s_n, r_d_n, label)
    rows = []
    for lo in (0.5, 2, 5, 10, 20, 40, 80):
        k = (dt >= lo) & (dt <= MAX_GAP)
        if k.sum() < 100: continue
        rows.append(dict(min_gap_min=lo, n=int(k.sum()),
                         other_bps=np.std(r_s[k]/np.sqrt(dt[k]))*1e4,
                         onchain_bps=np.std(r_d[k]/np.sqrt(dt[k]))*1e4,
                         ratio=vol_ratio(r_s[k]/np.sqrt(dt[k]), r_d[k]/np.sqrt(dt[k]))))
    sweep_df = pd.DataFrame(rows).set_index("min_gap_min")
    if verbose:
        print(f"{label}: {keep.sum():,} usable intervals of {len(keep):,} | "
              f"median gap {np.median(dt):.1f} min | observed time {dt.sum()/60/24:.1f} days")
        print(f"  ratio {res['ratio']:.2f}x (95% CI {res['ci_lo']:.2f} to {res['ci_hi']:.2f})")
    return r_s_n, r_d_n, dt, res, sweep_df

r_s_n, r_d_n, dt_cow, res_cow, sw_cow = matched("COW/ETH onchain", "COW/ETH syn",
                                                "COW/ETH synthetic vs on-chain")
print()
print(pd.DataFrame([res_cow]).round(4).to_string(index=False))
print("\nratio as short intervals are excluded (the on-chain analogue of the horizon sweep):")
display(sw_cow.round(3))
print(f"both columns fall, but the on-chain one falls faster: "
      f"{sw_cow.onchain_bps.iloc[0]:.1f} -> {sw_cow.onchain_bps.iloc[-1]:.1f} bps against "
      f"{sw_cow.other_bps.iloc[0]:.1f} -> {sw_cow.other_bps.iloc[-1]:.1f} bps, so the ratio rises "
      f"from {sw_cow.ratio.iloc[0]:.2f} to {sw_cow.ratio.iloc[-1]:.2f}.")
print(f"read the next markdown cell before quoting these: the trend is partly an artifact of the "
      f"{POOL_COW_FEE_BPS} bps pool fee.")

### Reading the sweep

SOL/ETH and COW/ETH move in opposite directions here, which is diagnostic rather than contradictory.

For SOL/ETH the ratio falls toward 1 as sampling coarsens. That is the signature of independent leg noise being swamped by genuine price movement.

For COW/ETH the ratio rises, from 1.3 at the finest matching to nearly 4 when only intervals longer than an hour are kept. Both volatilities fall as short intervals are dropped, but the on chain one falls faster, and the cause is selection. The pool reprices only when someone trades it, and with a 30 bps fee the correcting arbitrage is worth doing only once the gap clears that fee. A long quiet stretch between swaps is therefore itself evidence that the cross did not move, so conditioning on those stretches selects the calmest periods for the on chain series while the synthetic keeps measuring its own noise floor.

Both ends of that range are biased, in opposite directions. At the finest matching the on chain denominator is inflated by post swap bounce. At the coarsest it is deflated by the fee threshold and by selection. The COW/ETH distortion lies inside the band and this data cannot narrow it further. SOL/ETH is the figure to quote for the size of the construction effect, and the leg correlation below is the part that transfers across pairs.

## 7. WBETH/ETH, the case with three sources

This is the pair the original finding came from, and the only one here where a direct Binance listing, a Binance synthetic and an on-chain price all exist at once. That makes it the strongest test available. The Binance listing against the Binance synthetic is a controlled measurement of the construction, on one venue and one clock, exactly as for SOL/ETH. The on-chain series then says whether a third independent venue agrees.

wBETH is a staking receipt whose value against ETH drifts slowly upward as rewards accrue, so the true cross barely moves. That is what makes it the hard case rather than an easy one: the signal is small enough that any noise floor dominates it.

Two of the four series asked for are not available, and the reason is informative. There is no Uniswap V3 wBETH/ETH pool with usable volume on any chain, so the on-chain price is taken from PancakeSwap V3 on BNB, which carries 7,432 swaps over the window against 614 for the Ethereum PancakeSwap pool and none for Uniswap. An on-chain synthetic cannot be built at all: the only wBETH pools quoted against a stable are PancakeSwap BNB pairs that traded 180 and 60 dollars in total across two months, which is dust rather than a price feed. Note also that the PancakeSwap pool charges 5 bps against the Uniswap COW pool's 30, so it resolves movement far better, though still coarsely relative to this pair.

In [ ]:
last = pd.Timestamp(DAYS[-30])
W = {k: PX[k][PX[k].index >= last] for k in ("WBETHETH", "WBETH/ETH syn", "WBETH/ETH onchain")}
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=["WBETH/ETH from three sources",
                                    "the legs: WBETHUSDT and ETHUSDT, rebased to 100"])
for key, name, colr, wid in (
        ("WBETHETH",          "Binance WBETHETH (direct)",      C_DIR,     1.6),
        ("WBETH/ETH syn",     "Binance synthetic (WBETHUSDT / ETHUSDT)", C_SYN, 1.0),
        ("WBETH/ETH onchain", "PancakeSwap V3 on BNB (on-chain)", "#c22f2f", 1.0)):
    s = W[key].iloc[::3] if key != "WBETH/ETH onchain" else W[key]
    fig.add_trace(go.Scatter(x=s.index, y=s.to_numpy(), name=name,
                             line=dict(color=colr, width=wid)), row=1, col=1)
for leg, colr in (("WBETHUSDT", C_LEG_A), ("ETHUSDT", C_LEG_B)):
    s = PX[leg][PX[leg].index >= last].iloc[::3]
    fig.add_trace(go.Scatter(x=s.index, y=s.to_numpy()/s.iloc[0]*100, name=f"{leg} (rebased)",
                             line=dict(color=colr, width=1)), row=2, col=1)
fig.update_yaxes(title_text="ETH per wBETH", row=1, col=1)
fig.update_yaxes(title_text="legs rebased to 100", row=2, col=1)
style(fig, w=1150, h=650, title="WBETH/ETH: direct listing, synthetic, and on-chain pool, last 30 days")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
fig.show(renderer=RENDER)
save_html(fig, "chart_wbetheth_three_sources.html")

# controlled: Binance direct vs Binance synthetic, same venue and clock
rs_w, rd_w = aligned_returns("WBETHETH", "WBETH/ETH syn", 1)
res_wbeth = compare(rs_w, rd_w, "WBETH/ETH synthetic vs Binance direct")
print()
print(pd.DataFrame([res_wbeth]).round(4).to_string(index=False))
print(f"\nsynthetic is {res_wbeth['ratio']:.1f}x the direct volatility "
      f"(95% CI {res_wbeth['ci_lo']:.1f} to {res_wbeth['ci_hi']:.1f})")

sw_w = sweep("WBETHETH", "WBETH/ETH syn", "WBETH/ETH")
print("\nhorizon sweep, Binance direct vs Binance synthetic:")
display(sw_w.round(3))

# the on-chain pool against each Binance series, on matched timestamps
print()
_, _, _, res_oc_vs_dir, _ = matched("WBETH/ETH onchain", "WBETHETH",
                                    "WBETH/ETH Binance direct vs on-chain pool")
_, _, _, res_oc_vs_syn, _ = matched("WBETH/ETH onchain", "WBETH/ETH syn",
                                    "WBETH/ETH Binance synthetic vs on-chain pool")

## 8. The covariance decomposition

The synthetic log return is the COWUSDT return minus the ETHUSDT return, so

    Var(synthetic) = Var(A) + Var(B) - 2 rho sd(A) sd(B)

Solving for the rho that equates the synthetic variance with the observed direct variance gives the correlation the legs would need for the construction to be free. Where the required correlation exceeds the observed one, the shortfall is the excess the synthetic adds. For a genuine cross the legs share a common USD factor and much of their variance cancels. For a correlated pair the true cross barely moves, so the required correlation approaches 1, which independently quoted books cannot deliver.

In [ ]:
def decompose(leg_a, leg_b, direct_returns, minutes, label):
    a = PX[leg_a].resample(f"{minutes}min").last()
    b = PX[leg_b].resample(f"{minutes}min").last()
    d = pd.concat([a.rename("a"), b.rename("b")], axis=1).dropna()
    ra, rb = np.diff(np.log(d.a.to_numpy())), np.diff(np.log(d.b.to_numpy()))
    sa, sb = np.std(ra), np.std(rb)
    rho_obs = float(np.corrcoef(ra, rb)[0, 1])
    var_dir = float(np.var(direct_returns))
    rho_req = (sa**2 + sb**2 - var_dir)/(2*sa*sb)
    var_syn = sa**2 + sb**2 - 2*rho_obs*sa*sb
    return dict(case=label, leg_a_bps=sa*1e4, leg_b_bps=sb*1e4,
                rho_observed=rho_obs, rho_required=float(rho_req),
                syn_implied_bps=float(np.sqrt(max(var_syn, 0))*1e4),
                direct_bps=float(np.sqrt(var_dir)*1e4))

rs1, rd1 = aligned_returns("SOLETH", "SOL/ETH syn", 1)
dec = [decompose("SOLUSDT", "ETHUSDT", rd1, 1, "SOL/ETH @ 1 min")]
# COW: the direct variance is the matched-timestamp, per-sqrt-minute figure, so compare at 1 min
dec.append(decompose("COWUSDT", "ETHUSDT", r_d_n, 1, "COW/ETH @ 1 min equivalent"))
dec.append(decompose("WBETHUSDT", "ETHUSDT", rd_w, 1, "WBETH/ETH @ 1 min"))
D = pd.DataFrame(dec).set_index("case")
display(D.round(4))
for c in D.index:
    r = D.loc[c]
    verdict = ("achievable" if r.rho_required <= r.rho_observed + 0.05
               else "beyond what the legs deliver")
    print(f"{c}: legs correlate {r.rho_observed:.2f}, would need {r.rho_required:.2f} "
          f"for the synthetic to match ({verdict})")

## 9. What this costs a cap

The cap is the low quantile of price moves over the exclusivity window, so the quantity that matters is that tail quantile at T seconds rather than the standard deviation. Binance 1 minute data cannot resolve 26 seconds directly, so the figures below scale the 1 minute quantile by the square root of time, the bridge the clustering notebook measured at about 1.1 on liquid pairs.

The on chain COW series cannot support a 26 second quantile at all, so its row uses the same per unit time basis as Section 6.

In [ ]:
Q = 0.08                                    # the 8 percent target used in the cap notebooks
def tail_cap_bps(returns, minutes, T=T_EXCL):
    q = abs(np.quantile(returns, Q))*1e4
    return q*np.sqrt(T/(minutes*60))

rows = []
rs1, rd1 = aligned_returns("SOLETH", "SOL/ETH syn", 1)
rows.append(dict(pair="SOL/ETH", source="direct (Binance SOLETH)",
                 cap_bps=tail_cap_bps(rd1, 1)))
rows.append(dict(pair="SOL/ETH", source="synthetic (SOLUSDT/ETHUSDT)",
                 cap_bps=tail_cap_bps(rs1, 1)))
rows.append(dict(pair="WBETH/ETH", source="direct (Binance WBETHETH)",
                 cap_bps=tail_cap_bps(rd_w, 1)))
rows.append(dict(pair="WBETH/ETH", source="synthetic (WBETHUSDT/ETHUSDT)",
                 cap_bps=tail_cap_bps(rs_w, 1)))
rows.append(dict(pair="COW/ETH", source="on-chain (Uniswap V3, per-sqrt-min basis)",
                 cap_bps=tail_cap_bps(r_d_n, 1)))
rows.append(dict(pair="COW/ETH", source="synthetic (COWUSDT/ETHUSDT)",
                 cap_bps=tail_cap_bps(r_s_n, 1)))
cap = pd.DataFrame(rows)
cap["vs_direct"] = cap.groupby("pair").cap_bps.transform(
    lambda s: s/s.iloc[0] if s.iloc[0] > 0 else np.nan)     # undefined, not infinite, when direct is 0
display(cap.round(3))
print(f"caps are the {Q:.0%} quantile of 1 minute moves scaled to {T_EXCL}s by sqrt of time")
for p in cap.pair.unique():
    sub = cap[cap.pair == p]
    base, syn = sub.cap_bps.iloc[0], sub.cap_bps.iloc[1]
    if base > 0:
        print(f"  {p}: {syn:.2f} bps synthetic against {base:.2f} bps direct, {syn/base:.2f}x looser")
    else:
        print(f"  {p}: {syn:.2f} bps synthetic against a direct cap of exactly 0.00 bps. The pair is "
              f"tick pinned, so more than {Q:.0%} of 1 minute windows show no move at all. The ratio "
              f"is undefined rather than large, and no percentage overstatement describes it.")